In [22]:
import os
from dotenv import load_dotenv
from datasets import Dataset


In [23]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [79]:
#ragas module imports below
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/tmp/ipykernel_16917/1624149417.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness
/tmp/ipykernel_16917/1624149417.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness
/tmp/ipykernel_16917/1624149417.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrec

In [25]:
# Load .env file
load_dotenv()

True

In [26]:
# Verify API Key
if "GOOGLE_API_KEY" not in os.environ:
    print("❌ GOOGLE_API_KEY not found. Please check your .env file.")
else:
    print("✅ API Key Loaded Successfully")

✅ API Key Loaded Successfully


#### Building the Vector Knowledge Base

In [54]:
# 1. Prepare Data
docs = [
    Document(page_content="Paris is the capital and most populous city of France."),
    Document(page_content="Jane Austen was an English novelist known for Pride and Prejudice."),
    Document(page_content="The Great Wall of China is a series of fortifications."),
    Document(page_content="Mount Everest is Earth's highest mountain above sea level."),
    Document(page_content="Mike loves the color pink more than any other color.")
]

# 2. Setup Vector Store (Gemini Embeddings)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = FAISS.from_documents(docs, embeddings)
print(f"✅ Vector store created with {len(docs)} documents.")

✅ Vector store created with 5 documents.


#### Testing the Retrieval & Generation (RAG Pipeline)

In [57]:
# Initialize Gemini 1.5 Flash
# Use the newer, more capable version for 2026
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


def rag_pipeline(query):
    # Retrieval
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    context = [doc.page_content for doc in retrieved_docs]
    
    # Generation
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer only using the context provided:"
    response = llm.invoke(prompt)
    
    return response.content, context

# TEST RUN
test_query = "What is Mike's favorite color?"
answer, retrieved_context = rag_pipeline(test_query)

print(f"Query: {test_query}")
print(f"Retrieved Context: {retrieved_context}")
print(f"Generated Answer: {answer}")

Query: What is Mike's favorite color?
Retrieved Context: ['Mike loves the color pink more than any other color.', 'Jane Austen was an English novelist known for Pride and Prejudice.']
Generated Answer: Pink


#### Preparing the Evaluation Dataset

In [61]:
# Define test cases
questions = [
    "What is the capital of France?",
    "Who wrote Pride and Prejudice?",
    "What is Mike's favorite color?"
]
ground_truths = ["Paris", "Jane Austen", "Pink"]

# Build the dataset
data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

for q, gt in zip(questions, ground_truths):
    ans, ctx = rag_pipeline(q)
    data["question"].append(q)
    data["answer"].append(ans)
    data["contexts"].append(ctx)
    data["ground_truth"].append(gt)

eval_dataset = Dataset.from_dict(data)
print("✅ Evaluation Dataset Prepared")
print(eval_dataset.to_pandas().head())

✅ Evaluation Dataset Prepared
                         question       answer  \
0  What is the capital of France?        Paris   
1  Who wrote Pride and Prejudice?  Jane Austen   
2  What is Mike's favorite color?         Pink   

                                            contexts ground_truth  
0  [Paris is the capital and most populous city o...        Paris  
1  [Jane Austen was an English novelist known for...  Jane Austen  
2  [Mike loves the color pink more than any other...         Pink  


In [ ]:
#### Running Ragas Evaluation

In [80]:
import os
from google import genai  # This is the new 'google-genai' SDK
from ragas.llms import llm_factory

# 1. Initialize the Client (Replace .configure)
# The client automatically checks for GOOGLE_API_KEY in your env vars.
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# 2. Setup Ragas LLM wrapper
# Ragas expects the client object directly.
eval_llm = llm_factory(
    model="gemini-2.5-flash",
    provider="google",
    client=client
)

print("✅ Ragas Evaluator initialized with Gemini 2.5")

✅ Ragas Evaluator initialized with Gemini 2.5


In [ ]:
# 1. Wrap your Gemini models so Ragas recognizes them
# Assuming 'llm' is your ChatGoogleGenerativeAI and 'embeddings' is your GoogleGenerativeAIEmbeddings
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 2. Re-initialize metrics with these wrapped versions
faithfulness_metric = Faithfulness(llm=ragas_llm)
correctness_metric = AnswerCorrectness(llm=ragas_llm)
relevancy_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

# 3. Run evaluate with the wrapped components
print("🚀 Running Ragas evaluation...")
result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness_metric, relevancy_metric, correctness_metric],
    llm=ragas_llm, 
    embeddings=ragas_embeddings
)

/tmp/ipykernel_16917/2439964142.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_16917/2439964142.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


🚀 Running Ragas evaluation...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]